# Task 7: Creating a LG Summary Interactive Plot

## Importing Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.figure_factory as ff

## Main simulation function

In [ ]:
def logistic_func(y0, t, alpha, beta, delta, gamma, k):
    dxdt = (alpha * y0[0]*(1 - y0[0]/k)) - (beta * y0[0] * y0[1])
    dydt = (delta * y0[0] * y0[1]) - (gamma * y0[1])
    return dxdt, dydt

## Running the simulation

In [ ]:
# Prey, Predator
y0 = [10, 10]
alpha, beta = 0.1, 0.02
delta, gamma = 0.02, 0.4
end_time = 100
n_samples = 300
t = np.linspace(0, end_time, n_samples)
k = 120
y = odeint(logistic_func, y0, t, args=(alpha, beta, delta, gamma, k))

## Visualizing the results

In the **run_logistic_summary** function we: 
1. First create the direction fields for the Phase Space plot (this time the axis sizes are fixed). 
2. Then the quiver plot is created and added to the second subplot (plotly doesn’t currently support passing create_quiver as a normal trace). 
3. At this point, we can add a trace for the prey population, one for the predator population, and one for the Phase Space plot (prey vs predator spiral).
4. In order to make the chart animated, we then create a loop to generate a list of frames (one for each timestamp and for each trace, apart from the quiver plot).
5. Then different parameters are specified to in order to define how fast/slow the animation should be and where to place the different buttons.

In [ ]:
def run_logistic_summary(sim, y0):
    # Direction fields creation process from: https://scipy-cookbook.readthedocs.io/items/LoktaVolterraTutorial.html
    # Creating a grid and computing the direction at each point
    nb_points = 10
    x = np.linspace(0, 35, nb_points)
    y = np.linspace(0, 15, nb_points)
    X1 , Y1  = np.meshgrid(x, y) 
    # Computing growth rate on the grid
    DX1, DY1 = logistic_func([X1, Y1], 0, alpha, beta, delta, gamma, k)
    # Norm of the growth rate 
    M = (np.hypot(DX1, DY1))    
    # Avoiding zero division errors 
    M[ M == 0] = 1.0
    # Normalizing the arrows
    DX1 /= M      
    DY1 /= M

    force_field = ff.create_quiver(X1, Y1, DX1, DY1, scale=1, arrow_scale=0.5, name="Field Direction")

    fig = make_subplots(rows=1, cols=2)

    for d in force_field.data:
        fig.add_trace(go.Scatter(x=d['x'], y=d['y']),
                      row=1, col=2)

    fig.add_trace(
        go.Scatter(x=[i for i in range(len(sim))], y=sim[:, 0],
                         mode="lines",
                         line=dict(width=2, color="blue"), name='Prey'),
        row=1, col=1
    )

    fig.add_trace(
        go.Scatter(x=[i for i in range(len(sim))], y=sim[:, 1],
                         mode="lines",
                         line=dict(width=2, color="orange"), name='Predators'),
        row=1, col=1
    )

    fig.add_trace(go.Scatter(x=sim[:, 0], y=sim[:, 1], name="y0= " + str(y0)), row=1, col=2)


    frames =[go.Frame(
             data=[go.Scatter(
                x=[i for i in range(k)],
                y=sim[:, 0],
                mode="lines",
                line=dict(width=2, color="blue")),
                 go.Scatter(
                x=[i for i in range(k)],
                y=sim[:, 1],
                mode="lines",
                line=dict(width=2, color="orange")),
                go.Scatter(x=sim[:, 0][:k], y=sim[:, 1][:k], name="y0= " + str(y0)),
                go.Scatter(x=force_field.data[0]['x'], y=force_field.data[0]['y'])
                ],
                traces=[1,2,3,0])
             for k in range(len(sim))] 

    fig.frames=frames
    fig.update_layout(
        updatemenus= [
                {
                    "buttons": [
                        {
                            "args": [None, {"frame": {"duration": 10, "redraw": False},
                                            "fromcurrent": True, 
                                            "transition": {"duration": 1,
                                                           "easing": "quadratic-in-out"}}],
                            "label": "Play",
                            "method": "animate"
                        },
                        {
                            "args": [[None], {"frame": {"duration": 0, "redraw": False},
                                              "mode": "immediate",
                                              "transition": {"duration": 0}}],
                            "label": "Pause",
                            "method": "animate"
                        }
                    ],
                    "direction": "left",
                    "pad": {"r": 10, "t": 87},
                    "showactive": False,
                    "type": "buttons",
                    "x": 0.13,
                    "xanchor": "right",
                    "y": 1.4,
                    "yanchor": "top"
                }
            ]
    )

    fig.update_xaxes(title_text="Simulation Steps", row=1, col=1)
    fig.update_yaxes(title_text="Population Size", row=1, col=1)
    fig.update_xaxes(title_text="Number of Prey", row=1, col=2)
    fig.update_yaxes(title_text="Number of Predators", row=1, col=2)
    fig.update_layout(height=600, width=1200, title_text="Simulation Report")
    fig.show()
    
run_logistic_summary(y, y0)